# ASRQuant 1.3 — Public Practical Case
## ECB EUR curve → 5Y IRS → curve risk → validation → audit trail

This notebook demonstrates the main ASRQuant 1.3 workflows inside one coherent quantitative-finance case.

**Questions**
1. What changed in the EUR curve?
2. How does that affect a 5Y payer IRS?
3. How do we stress and explain the position?
4. Can a simple curve research idea survive selection-risk diagnostics?
5. Can another researcher reproduce the result?

> The ECB AAA sovereign spot curve is a transparent public research proxy. Production EUR IRS work should use appropriate OIS discounting, projection curves, collateral conventions and live market swap quotes.

## Install

```bash
pip install --upgrade "asrquant[providers,visualization]"
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import asrquant as asr

assert asr.__version__.startswith("1.3"), f"Expected ASRQuant 1.3.x, got {asr.__version__}"
OUT = Path("asrquant_public_case_output")
OUT.mkdir(exist_ok=True)
print("ASRQuant:", asr.__version__)

## 1. Load, validate and freeze the EUR curve history

In [ ]:
MATS = ("3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","15Y","20Y","30Y")
YEARS = np.array([0.25,0.5,1,2,3,5,7,10,15,20,30], dtype=float)

def fallback():
    rng = np.random.default_rng(130)
    idx = pd.bdate_range("2025-01-02", periods=320)
    base = np.array([.021,.0215,.022,.023,.024,.0255,.0265,.0275,.0285,.029,.0295])
    level = np.cumsum(rng.normal(0, 8e-5, len(idx)))[:,None]
    slope = np.cumsum(rng.normal(0, 2.5e-5, len(idx)))[:,None]
    values = base + level + slope*np.linspace(-1,1,len(base))[None,:] + rng.normal(0,3e-5,(len(idx),len(base)))
    return pd.DataFrame(values, index=idx, columns=MATS)

try:
    curve_history = asr.ECBProvider().yield_curve_history(
        maturities=MATS, start="2024-01-01"
    ).dropna()
    DATA_MODE = "LIVE ECB DATA"
except Exception:
    curve_history = fallback()
    DATA_MODE = "OFFLINE DEMO FALLBACK"

if getattr(curve_history.index, "tz", None) is not None:
    curve_history.index = curve_history.index.tz_convert(None)

quality = asr.data.validate(curve_history)
store = asr.data.DataStore(OUT / "data_store")
snapshot = store.put(
    "eur_curve_history", curve_history,
    source="ECB" if DATA_MODE == "LIVE ECB DATA" else "offline-demo",
    metadata={"data_mode": DATA_MODE, "units": "decimal per annum"},
)

print(DATA_MODE, "| rows:", len(curve_history), "| clean:", quality.is_clean)
print("fingerprint:", snapshot.hash)
display(quality.summary)

## 2. Read the market: curve history + PCA

In [ ]:
ax = (curve_history[["2Y","5Y","10Y","30Y"]]*100).plot(
    figsize=(10,4), title=f"EUR curve history — {DATA_MODE}"
)
ax.set_ylabel("Yield (%)")
plt.show()

changes_bp = curve_history.diff().dropna()*10_000
pca = asr.factors.pca(changes_bp, n_components=3, standardize=False, component_prefix="PC")
display(pca.summary.round(4))
display(pca.loadings.round(4))

## 3. Build two curve states and create a 5Y payer IRS

In [ ]:
r0, r1 = curve_history.iloc[-2], curve_history.iloc[-1]
d0, d1 = pd.Timestamp(r0.name), pd.Timestamp(r1.name)

curve0 = asr.rates.DiscountCurve.from_zero_rates(
    YEARS, r0.to_numpy(float), interpolation="log_linear",
    name=f"EUR {d0.date()}", metadata={"source": DATA_MODE}
)
curve1 = asr.rates.DiscountCurve.from_zero_rates(
    YEARS, r1.to_numpy(float), interpolation="log_linear",
    name=f"EUR {d1.date()}", metadata={"source": DATA_MODE}
)

lab = asr.RateQuantLab(curve0)
par5 = lab.par_swap(0.0, 5.0, frequency=2)

swap = asr.rates.InterestRateSwap(
    maturity=5.0, fixed_rate=par5, notional=10_000_000,
    fixed_frequency=2, float_frequency=2, position="payer"
)

pv0, pv1 = swap.price(curve0), swap.price(curve1)
print("t0:", d0.date(), "| t1:", d1.date())
print(f"5Y par rate: {par5:.4%}")
print(f"PV t0: EUR {pv0:,.2f} | PV t1: EUR {pv1:,.2f}")

## 4. DV01, key-rate DV01 and quote-space PV01

In [ ]:
risk = asr.rates.curve_risk(
    swap, curve0, key_maturities=[1,2,5,10,30]
)
display(risk.summary.round(4))
display(risk.key_rate_dv01.rename("EUR / bp").to_frame().round(2))

deposit_6m = (1.0 / float(curve0.df(0.5)) - 1.0) / 0.5
swap_quotes = {m: lab.par_swap(0.0, m, frequency=2) for m in [1,2,3,5,7,10]}

build = asr.rates.CurveBuilder.from_quotes(
    deposits={0.5: deposit_6m},
    swaps=swap_quotes,
    swap_frequency=2,
    interpolation="log_linear",
).build()

quote_risk = asr.rates.curve_risk(
    swap, build.curve, key_maturities=[1,2,5,10], build_result=build
)
display(build.repricing_errors)
display(quote_risk.quote_pv01.rename("EUR / bp quote move").to_frame().round(2))

## 5. Explain realized P&L and run deterministic stresses

In [ ]:
explain = asr.rates.pnl_explain(swap, curve0, curve1)
display(explain.contributions.rename("EUR").to_frame().round(2))

scenarios = {
    "Parallel +25bp": asr.scenarios.parallel_rate_shift(25),
    "Steepener +25bp": asr.scenarios.steepener(25),
    "Flattener +25bp": asr.scenarios.flattener(25),
}
rows = {}
for name, sc in scenarios.items():
    res = asr.scenarios.run(sc, instrument=swap, curve=curve0)
    rows[name] = {"base":res.base_value, "shocked":res.shocked_value, "pnl":res.pnl}

scenario_table = pd.DataFrame(rows).T
display(scenario_table.round(2))
scenario_table["pnl"].plot(kind="bar", figsize=(8,3), title="Scenario P&L (EUR)")
plt.show()

## 6. Research validation on the same real curve history

**Research question:** can an extreme 2s10s slope deviation contain information about the next observed 5Y yield move?

We intentionally test several lookbacks. The output below is a **research proxy**, not a tradeable IRS return series.

In [ ]:
slope = curve_history["10Y"] - curve_history["2Y"]
move5_bp = curve_history["5Y"].diff()*10_000
specs = {}

for w in [20,40,60,120]:
    z = ((slope - slope.rolling(w).mean()) / slope.rolling(w).std()).shift(1)
    signal = -np.tanh(z)
    specs[f"MR_{w}d"] = signal * move5_bp * 1e-4

strategy_returns = pd.DataFrame(specs).dropna()
display(strategy_returns.describe().T.round(6))

splits = list(asr.validation.combinatorial_purged_cv_splits(
    n_samples=len(strategy_returns), n_groups=6, test_groups=2, embargo=5
))
pbo = asr.validation.probability_of_backtest_overfitting(strategy_returns)
reality = asr.validation.reality_check(strategy_returns)
spa = asr.validation.spa_test(strategy_returns)

print("CPCV splits:", len(splits))
print("PBO"); display(getattr(pbo, "summary", pbo))
print("Reality Check"); display(getattr(reality, "summary", reality))
print("SPA"); display(getattr(spa, "summary", spa))

## 7. Register the experiment and export a report

The final result keeps the **data fingerprint + configuration + quantitative artifacts** together.

In [ ]:
experiment = asr.research.Experiment(
    name="ASRQuant 1.3 — EUR Rates Public Case",
    config={
        "instrument":"5Y payer IRS",
        "notional_eur":10_000_000,
        "curve_source":DATA_MODE,
        "research_lookbacks":[20,40,60,120],
        "validation":["CPCV","PBO","Reality Check","SPA"],
    },
    data={"eur_curve_history": snapshot},
    seed=130,
    tags=("ASRQuant-1.3","public","rates","validation"),
)

registry = asr.research.ExperimentRegistry(OUT / "experiments.jsonl")
registry.add(experiment)

report = asr.research.ResearchReport("ASRQuant 1.3 — EUR Rates Public Case")
for obj, name in [
    (experiment,"experiment"),
    (snapshot,"data_snapshot"),
    (pca,"curve_pca"),
    (risk,"swap_curve_risk"),
    (explain,"pnl_explain"),
]:
    report.add(obj, name)

html = report.export(OUT / "public_case_report.html")
json_path = report.export(OUT / "public_case_report.json")

display(report.audit())
print("Experiment fingerprint:", experiment.fingerprint)
print("HTML:", html)
print("JSON:", json_path)

# Takeaway

This single case used the main ASRQuant 1.3 ideas:

**Data → Curve → Instrument → Risk → P&L Explain → Stress → Validation → Reproducibility**

ASRQuant is not designed to remove financial judgement. It is designed to make the assumptions, intermediate objects, validation evidence and research lineage easier to inspect and challenge.